In [77]:
import os
import sys
from dotenv import load_dotenv
from neo4j import GraphDatabase

from langchain_openai import OpenAIEmbeddings
from neo4j_graphrag.retrievers import HybridCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

from pprint import pprint

load_dotenv()

True

In [2]:
URI = os.environ["NEO4J_URI"]
AUTH = (os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"])
driver = GraphDatabase.driver(URI, auth=AUTH)

# Retriever

In [4]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    base_url=os.environ["SUMOPOD_URI"] + "/v1",
    api_key=os.environ["SUMOPOD_KEY"]
)

In [74]:
def fmt(record):
    pasal_id = record.get("pasal_id") or ""
    nomor = record.get("nomor") or pasal_id.split(":")[-1]
    uu = record.get("uu") or ""
    jenis = record.get("jenis") or ""

    # --- content: yang dibaca LLM ---
    label = f"UU {uu}" if uu else f"UU {pasal_id.split(':')[0]}"
    bagian = [f"### Pasal {nomor} — {label}"]

    teks = (record.get("teks") or "").strip()
    bagian.append(teks)

    penj = (record.get("penjelasan") or "").strip()
    if penj and penj.lower() != "cukup jelas":
        bagian.append(f"[Penjelasan resmi] {penj}")

    for d in (record.get("diubah") or []):
        if not d or not d.get("teks"):
            continue
        bagian.append(
            f"[VERSI TERBARU — Pasal {d.get('target') or nomor} "
            f"{d.get('aksi')} oleh Ciptaker {d.get('dari')}]\n{d['teks'].strip()}"
        )

    rujuk = record.get("rujukan") or []
    if rujuk:
        bagian.append(f"[Kaitan: {', '.join(rujuk)}]")

    content = "\n".join(bagian)

    # --- metadata: debug/tampilan ---
    metadata = {
        "pasal_id": pasal_id,
        "nomor": nomor,
        "uu": uu,
        "jenis": jenis,
        "score": record.get("score"),
        "ada_perubahan": bool([d for d in (record.get("diubah") or []) if d and d.get("teks")]),
    }
    return RetrieverResultItem(content=content, metadata=metadata)


In [76]:
_ret = HybridCypherRetriever(
    driver,
    "pasal_embedding",
    "pasal_keyword",
    retrieval_query="""
        WITH node AS anchor, score
        OPTIONAL MATCH (anchor)-[:BAGIAN_DARI]->(u:UU)
        OPTIONAL MATCH (anchor)<-[e:MENGUBAH|MENGHAPUS|MENYISIPKAN]-(ubah:Pasal)
        OPTIONAL MATCH (anchor)-[:MERUJUK]->(rujuk:Pasal)
        RETURN anchor.text_full AS teks,
            anchor.penjelasan AS penjelasan,
            anchor.pasal_id AS pasal_id,
            score,
            coalesce(u.judul,'') AS uu,
            collect(DISTINCT {
                aksi: e.aksi,
                teks: e.teks_baru,
                dari: ubah.pasal_id,
                target: e.pasal_target
            }) AS diubah,
            collect(DISTINCT rujuk.pasal_id)[0..3] AS rujukan
    """,
    embedder=embeddings,
    result_formatter=fmt
)
def retriever(question, top_k = 3):
    raw = _ret.search(query_text=question, top_k= top_k)
    
    return {
        "question": question,
        "docs": [
            {"pasal_id": i.metadata["pasal_id"], "score": i.metadata["score"],
            "content": i.content}
            for i in raw.items
        ],
        "context": "\n\n---\n\n".join(i.content for i in raw.items),
    }


In [54]:
pprint(retriever("regulasi BPJS"))

('### Pasal 9 — UU CIPTA KERJA\n'
 '(1) BPJS Kesehatan sebagaimana dimaksud dalam\n'
 'Pasal 5 ayat (21 huruf a\n'
 'berfungsi\n'
 'menyelenggarakan program jaminan kesehatan.\n'
 '(21 BPJS Ketenagakerjaan sebagaimana dimaksud\n'
 'dalam Pasal 5 ayat (2) huruf b berfungsi\n'
 'menyelenggarakan program jaminan kecelakaan\n'
 'kerja, program jaminan kematian, program\n'
 'jaminan pensiun, program jaminan hari tua, dan\n'
 'program jaminan kehilangan pekerjaan.\n'
 '[Kaitan: 24/2011:5, 24/2011:9]\n'
 '\n'
 '---\n'
 '\n'
 '### Pasal 6 — UU CIPTA KERJA\n'
 '(1) BPJS Kesehatan sebagaimana dimaksud dalam\n'
 'Pasal 5 ayat (2) huruf a menyelenggarakan program\n'
 'jaminan kesehatan.\n'
 '(2) BPJS Ketenagakerjaan sebagaimana dimaksud\n'
 'dalam Pasal 5 ayat (2) huruf b menyelenggarakan\n'
 'program:\n'
 'a- jaminan kecelakaan kerja;\n'
 'b. jaminan hari tua;\n'
 'c. jaminan pensiun;\n'
 'd. jaminan kematian; dan\n'
 'e. jaminan kehilangan pekerjaan.\n'
 '[Kaitan: 24/2011:5, 24/2011:6]\n'
 '\n'


In [55]:
llm = ChatOpenAI(
    model="gemini-3.8-flash",
    base_url=os.environ["9ROUTER_URI"] + "/v1",
    api_key=os.environ["9ROUTER_KEY"],
    temperature=0,
)

In [56]:
system_prompt = """
Jawab HANYA dari konteks di bawah. Kalau konteks tak memuat jawabannya, 
bilang tidak tahu. Bila ada blok [VERSI TERBARU], itu hukum yang berlaku — 
pakai itu, dan sebut pasal lama hanya sebagai perbandingan.
"""
human_prompt = """
Pertanyaan:
{question}
Konteks:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_prompt),
    HumanMessagePromptTemplate.from_template(human_prompt)
])

In [78]:
# chain = (
#     {"question": RunnablePassthrough(), "context": retriever}
#     | prompt
#     | llm
#     | StrOutputParser()
# )

chain = (
    RunnableLambda(retriever) 
    |RunnablePassthrough.assign(
        answer=prompt | llm | StrOutputParser()
    )
)

In [92]:
q1 = chain.invoke("pekerjaan apa aja yang boleh kontrak?")

In [104]:
pprint(q1['answer'])

('Berdasarkan ketentuan yang berlaku saat ini (**Pasal 59 ayat (1) Ciptaker '
 '2/2022 [VERSI TERBARU]**), pekerjaan yang boleh dibuat perjanjian kerja '
 'waktu tertentu (kontrak) adalah pekerjaan yang selesai dalam waktu tertentu, '
 'yaitu:\n'
 '\n'
 '1. Pekerjaan yang sekali selesai atau yang sementara sifatnya;\n'
 '2. Pekerjaan yang diperkirakan penyelesaiannya dalam waktu yang tidak '
 'terlalu lama;\n'
 '3. Pekerjaan yang bersifat musiman;\n'
 '4. Pekerjaan yang berhubungan dengan produk baru, kegiatan baru, atau produk '
 'tambahan yang masih dalam percobaan atau penjajakan; atau\n'
 '5. Pekerjaan yang jenis dan sifat atau kegiatannya bersifat tidak tetap.\n'
 '*(Pasal 59 ayat (2) menegaskan pekerjaan yang bersifat tetap tidak boleh '
 'dikontrak).*\n'
 '\n'
 '**Perbandingan dengan versi lama (Pasal 59 ayat (1) UU Ketenagakerjaan):**\n'
 '* Huruf b pada versi lama membatasi jangka waktu *"paling lama 3 (tiga) '
 'tahun"*, sedangkan di versi terbaru batasan 3 tahun tersebut dih

In [94]:
for d in q1['docs']:
    pprint(d['content'])

('### Pasal 59 — UU Ketenagakerjaan\n'
 '(1) Perjanjian kerja untuk waktu tertentu hanya dapat dibuat untuk pekerjaan '
 'tertentu yang menurut jenis dan sifat atau kegiatan pekerjaannya akan '
 'selesai dalam waktu tertentu, yaitu : a. pekerjaan yang sekali selesai atau '
 'yang sementara sifatnya; b. pekerjaan yang diperkirakan penyelesaiannya '
 'dalam waktu yang tidak terlalu lama dan paling lama 3 (tiga) tahun; c. '
 'pekerjaan yang bersifat musiman; atau d. pekerjaan yang berhubungan dengan '
 'produk baru, kegiatan baru, atau produk tambahan yang masih dalam percobaan '
 'atau penjajakan. (2) Perjanjian kerja untuk waktu tertentu tidak dapat '
 'diadakan untuk pekerjaan yang bersifat tetap. (3) Perjanjian kerja untuk '
 'waktu tertentu dapat diperpanjang atau diperbaharui. (4) Perjanjian kerja '
 'waktu tertentu yang didasarkan atas jangka waktu tertentu dapat diadakan '
 'untuk paling lama 2 (dua) tahun dan hanya boleh diperpanjang 1 (satu) kali '
 'untuk jangka waktu paling la

In [95]:
q2 = chain.invoke("aturan outsourcing")

In [100]:
for d in q2['docs']:
    pprint(d['content'])

('### Pasal 128 — UU Ketenagakerjaan\n'
 'Dalam hal perjanjian kerja tidak memuat aturan-aturan yang diatur dalam '
 'perjanjian kerja bersama maka yang berlaku adalah aturan-aturan dalam '
 'perjanjian kerja bersama.')
('### Pasal 64 — UU Ketenagakerjaan\n'
 'Perusahaan dapat menyerahkan sebagian pelaksanaan pekerjaan kepada '
 'perusahaan lainnya melalui perjanjian pemborongan pekerjaan atau penyediaan '
 'jasa pekerja/buruh yang dibuat secara tertulis.\n'
 '[VERSI TERBARU — Pasal 64 diubah oleh Ciptaker 2/2022:81:64]\n'
 '(1) Perusahaan dapat menyerahkan sebagian\n'
 'pelaksanaan pekerjaan kepada Perusahaan\n'
 'lainnya melalui perjanjian alih daya yang dibuat\n'
 'secara tertulis.\n'
 '(2) Pemerintah menetapkan sebagian pelaksanaan\n'
 'pekerjaan sebagaimana dimaksud pada ayat (1).\n'
 '(3) Ketentuan lebih lanjut mengenai penetapan\n'
 'sebagian pelaksanaan pekerjaan sebagaimana\n'
 'dimaksud pada ayat (21 diatur dalam Peraturan\n'
 'Pemerintah.')
('### Pasal 66 — UU Ketenagakerjaa

In [101]:
q3 = chain.invoke("bagaimana ak cuti karyawan?")

In [102]:
pprint(q3['answer'])

('Berdasarkan **[VERSI TERBARU]** (Pasal 79 UU Cipta Kerja):\n'
 '\n'
 '1. **Hak Cuti Tahunan**: Wajib diberikan kepada pekerja/buruh paling sedikit '
 '**12 hari kerja** setelah bekerja selama **12 bulan secara terus-menerus** '
 '(ayat 3).\n'
 '2. **Pelaksanaan**: Diatur dalam Perjanjian Kerja, Peraturan Perusahaan, '
 'atau Perjanjian Kerja Bersama (ayat 4).\n'
 '3. **Istirahat Panjang**: Perusahaan tertentu *dapat* memberikan istirahat '
 'panjang yang diatur dalam Perjanjian Kerja, Peraturan Perusahaan, atau '
 'Perjanjian Kerja Bersama (ayat 5).\n'
 '\n'
 '---\n'
 '**Perbandingan dengan pasal lama (UU Ketenagakerjaan No. 13/2003):**\n'
 'Ketentuan cuti tahunan tetap sama (minimal 12 hari setelah 12 bulan kerja). '
 'Namun, pada aturan lama, ketentuan hak istirahat panjang diatur secara '
 'eksplisit dan rinci minimal 2 bulan pada tahun ke-7 dan ke-8 bagi yang telah '
 'bekerja 6 tahun terus-menerus (Pasal 79 ayat 2 huruf d), sedangkan pada '
 'aturan terbaru pelaksanaan istirahat

In [103]:
for d in q3['docs']:
    pprint(d['content'])

('### Pasal 79 — UU CIPTA KERJA\n'
 '(1) Pengusaha wajib memberi:\n'
 'a- waktu istirahat; dan\n'
 'b. cuti.\n'
 '(21 Waktu istirahat sebagaimana dimaksud pada ayat\n'
 '(1) huruf a wajib diberikan kepada Pekerja/Buruh\n'
 'paling sedikit meliputi:\n'
 'a. istirahat antara jam kerja, paling sedikit\n'
 'setengah jam setelah bekerja selama 4 (empat)\n'
 'jam terus-menerus, dan waktu istirahat\n'
 'tersebut tidak termasuk jam kerja; dan\n'
 'b. istirahat mingguan 1 (satu) hari untuk 6\n'
 '(enam) hari kerja dalam I (satu) minggu.\n'
 '(3) Cuti sebagaimana dimaksud pada ayat (1) huruf b\n'
 'yang wajib diberikan kepada Pekerja/Buruh, yaitu\n'
 'cuti tahunan, paling sedikit 12 (dua belas) hari\n'
 'kerja setelah Pekerja/Buruh yang bersangkutan\n'
 'bekerja selama 12 (dua belas) bulan secara terus\n'
 'menerus.\n'
 '(4) Pelaksanaan cuti tahunan sebagaimana dimaksud\n'
 'pada ayat (3) diatur dalam Perjanjian Kerja,\n'
 'Peraturan Perusahaan, atau Perjanjian Kerja\n'
 'Bersama.\n'
 '(5) Selai